# ROC and precision-recall plots

This notebook runs the standalone plotting scripts in `scripts/roc_pr_plot_scripts/`. The plotting and metric logic remains in those Python files rather than being duplicated in the notebook.

The three analyses compare:

1. activity-inference methods,
2. prior networks, and
3. network-weighting strategies.


## Requirements and outputs

Run `run_methods.ipynb` first so that the required activity-score parquet files exist. Input and output locations are configured at the top of each plotting script. By default, the scripts read:

- `scRNASeq/<dataset>.h5ad`
- `scores/<dataset>/*.parquet`
- `common_tfs/common_tfs_<prior>.tsv`

Plots and summary metrics are written to the configured output roots:

- `Methods_ROC_PR_Plots/`
- `Priors_ROC_PR_Plots/`
- `Weights_ROC_PR_Plots/`

Within each output root, ROC and PR curves are organized as `<dataset>/roc/<TF>_roc.svg` and `<dataset>/pr/<TF>_pr.svg`, and the corresponding `tf_curve_metrics_*.tsv` summary is saved at the output-root level.

Datasets or score combinations that are not available locally are reported and skipped. Add the remaining files later and rerun the relevant cells.


## Setup

The runner works when Jupyter is started from either the repository root or this notebook's directory. Console output from each subprocess is streamed into its notebook cell.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_analysis_dir(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "utility_functions.py").is_file():
            return candidate
        nested = candidate / "reproduce" / "Reproduce scRNASeq Results"
        if (nested / "scripts" / "utility_functions.py").is_file():
            return nested
    raise FileNotFoundError("Could not locate the Reproduce scRNASeq Results directory")

analysis_dir = find_analysis_dir(Path.cwd().resolve())
scripts_dir = analysis_dir / "scripts" / "roc_pr_plot_scripts"

matplotlib_cache = analysis_dir / ".cache" / "matplotlib"
matplotlib_cache.mkdir(parents=True, exist_ok=True)

def run_plot_script(filename: str) -> None:
    script_path = scripts_dir / filename
    if not script_path.is_file():
        raise FileNotFoundError(f"Missing plotting script: {script_path}")

    environment = os.environ.copy()
    environment.setdefault("MPLCONFIGDIR", str(matplotlib_cache))

    print(f"Running {script_path.name}")
    print(f"Working directory: {analysis_dir}")
    process = subprocess.Popen(
        [sys.executable, str(script_path)],
        cwd=analysis_dir,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is not None:
        for line in process.stdout:
            print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, [sys.executable, str(script_path)])

    print(f"Completed {script_path.name}")

print(f"Analysis directory: {analysis_dir}")
print(f"Plot scripts directory: {scripts_dir}")


## 1. Compare activity-inference methods

Generate ROC and precision-recall plots comparing z-aggregate, VIPER, ULM, and z-score with the CausalPath prior and uniform weights.


In [ ]:
run_plot_script("Methods_ROC_PR_Plot.py")


## 2. Compare prior networks

Generate ROC and precision-recall plots comparing CausalPath, CollecTRI, DoRothEA, and ensemble prior networks with matched TFs.


In [ ]:
run_plot_script("Priors_ROC_PR_Plot.py")


## 3. Compare network-weighting strategies

Generate ROC and precision-recall plots comparing uniform, correlation, specificity, and non-zero-rate weighting for z-aggregate.


In [ ]:
run_plot_script("Weights_ROC_PR_Plot.py")
